# Research Study: Patient-age trust and controlled processing benchmark
**Authors:** Sharipkhan Merey; Olzhabek Marzhan.

All research answers in this demonstration are **synthetic**. Repeated records in load testing are **not extra patients**. Pipeline timing data were measured in a separate execution environment; no human data were collected.

In [1]:
from pathlib import Path
import sys, json, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from process_data import analyze
from benchmark_statistics import read_logs, aggregate, save_summary
from benchmark_visualize import figures
from visualize import plot
print("Project:", ROOT.name); print("Random seed: 42")

Project: AI_Medical_Trust_Research
Random seed: 42


## 1. Import synthetic survey and recorded JSONL, filter warm-ups
The 45 survey rows are demonstration-only. We filter `phase == benchmark` for all performance statistics.

In [2]:
survey = analyze(ROOT / "data/sample/sample_synthetic.csv")
logs = read_logs(ROOT / "data/raw/test_run.jsonl")
log_df = pd.DataFrame(logs)
print("Synthetic survey records:", survey["total"], "valid:", survey["valid"], "invalid:", survey["invalid"])
print("Raw log rows:", len(log_df), "warmups:", int((log_df.phase=="warmup").sum()), "evaluation:", int((log_df.phase=="benchmark").sum()))
assert set(log_df.phase) == {"warmup", "benchmark"}
assert set(log_df.method) == {"baseline", "proposed"}
assert sorted(log_df.load_records.unique()) == [45,90,180,360]
print(log_df.groupby(["load_records", "method", "phase"]).size().to_string())

Synthetic survey records: 45 valid: 45 invalid: 0
Raw log rows: 720 warmups: 80 evaluation: 640
load_records  method    phase    
45            baseline  benchmark    80
                        warmup       10
              proposed  benchmark    80
                        warmup       10
90            baseline  benchmark    80
                        warmup       10
              proposed  benchmark    80
                        warmup       10
180           baseline  benchmark    80
                        warmup       10
              proposed  benchmark    80
                        warmup       10
360           baseline  benchmark    80
                        warmup       10
              proposed  benchmark    80
                        warmup       10


## 2. Median/p25/p75, p95, p99, 95% CI and error rate
Each cell below uses **80 measured executions**, with ten explicitly marked warm-up runs excluded. We bootstrap the median using 2,000 resamples with fixed seed 42.

In [3]:
summary = aggregate(logs)
summary_path = save_summary(summary, ROOT / "data/processed/metrics_summary.csv")
metrics = pd.DataFrame(summary)
print(metrics.to_string(index=False))
print("Saved:", summary_path.relative_to(ROOT))
assert all(metrics.warmup_runs_excluded == 10)
assert all(metrics.benchmark_runs == 80)
assert all(metrics.ci95_low_ms <= metrics.median_ms) and all(metrics.median_ms <= metrics.ci95_high_ms)

  method  load_records  warmup_runs_excluded  benchmark_runs  successful_runs  error_rate  median_ms   p25_ms   p75_ms   p95_ms   p99_ms  ci95_low_ms  ci95_high_ms
baseline            45                    10              80               80         0.0   0.074408 0.073799 0.075326 0.137113 0.207546     0.074017      0.074673
proposed            45                    10              80               80         0.0   0.040932 0.040711 0.041272 0.054200 0.073710     0.040872      0.040982
baseline            90                    10              80               80         0.0   0.127343 0.126283 0.131471 0.188881 0.261221     0.126797      0.128033
proposed            90                    10              80               80         0.0   0.069520 0.068919 0.070234 0.098975 2.380817     0.069155      0.069770
baseline           180                    10              80               80         0.0   0.239001 0.226978 0.329757 0.492062 1.094020     0.229828      0.255376
proposed        

## 3. Baseline vs proposed implementation
**Baseline:** three independent scans to compute cohort counts/means. **Proposed:** one pass; the two methods agree exactly in test cases. This measures synthetic survey-processing operations, **not** clinical model response time.

In [4]:
from compare_methods import load_rows, repeat_workload, baseline, proposed
rows = load_rows(ROOT / "data/sample/sample_synthetic.csv")
for factor in (1,2,4,8):
    arr=repeat_workload(rows,factor)
    assert baseline(arr)==proposed(arr)
print("Identical method outputs at 45, 90, 180 and 360 repeated synthetic records.")
print(pd.DataFrame(survey["summary"]).to_string(index=False))

Identical method outputs at 45, 90, 180 and 360 repeated synthetic records.
age_group  n  mean_trust  median_trust  sd_trust  ci95_lower  ci95_upper  prior_ai_yes
    18-30 15       3.867             4     0.834       3.467       4.267             5
    31-50 15       3.200             3     1.014       2.733       3.667             7
      51+ 15       3.067             3     0.799       2.667       3.467             6


## 4. Publication figures and PDF/PNG export
Error bars indicate 95% bootstrap CIs for **median latency**, not uncertainty about the synthetic population. Survey plots show mean and median CIs separately.

In [5]:
plot(str(ROOT / "data/sample/sample_synthetic.csv"),str(ROOT / "reports/figures"))
figures(ROOT / "reports/figures")
for name in ("fig1_latency_vs_load", "fig2_distribution_boxplot", "fig1_trust_by_age", "fig2_trust_distribution"):
    assert (ROOT / "reports/figures" / (name+".pdf")).exists()
    assert (ROOT / "reports/figures" / (name+".png")).exists()
    print("Saved:",name,"PDF + 350-DPI PNG")

Saved: fig1_latency_vs_load PDF + 350-DPI PNG
Saved: fig2_distribution_boxplot PDF + 350-DPI PNG
Saved: fig1_trust_by_age PDF + 350-DPI PNG
Saved: fig2_trust_distribution PDF + 350-DPI PNG


## 5. Interpretation and limitations
The proposed implementation avoids repeated scans and was faster in this controlled execution environment. The benchmarks are based on **synthetic repeated input rows**, cannot establish performance on unseen data or other computers, and do not measure AI recommendation quality. No real patient responses or hypothesis tests are reported.

[README](../README.md) · [C4 domain model](../docs/domain_model.md) · [PDF report](../reports/Patient_Age_Trust_Report.pdf)